# Notion-Zotero Analysis

Full pipeline: load canonical bundles → build summary tables → clean → explore.

**Data sources** (pick one):
-  — bundles pulled via 
-  — bundles pulled via 
-  — bundles parsed from local fixture files

In [4]:
from pathlib import Path
import pandas as pd

from notion_zotero.analysis import (
    load_canonical_records,
    build_summary_dataframes,
    clean_table,
    run_analysis,
)

# Load from live pulled data
pulled_dir = Path("data/pulled/notion/learning_analytics_review")
if not pulled_dir.exists() or not any(pulled_dir.glob("*.canonical.json")):
    raise FileNotFoundError(
        f"No pulled data found in {pulled_dir}. "
        "Run: notion-zotero pull-notion"
    )

bundles = load_canonical_records(str(pulled_dir))
print(f"Loaded {len(bundles)} bundles from {pulled_dir}")

Loaded 442 bundles from data\pulled\notion\learning_analytics_review


In [5]:
# --- Cell 2: Inspect loaded bundles ---
print(f"Loaded {len(bundles)} canonical bundles")

if bundles:
    sample = bundles[0]
    refs = sample.get("references", [])
    if refs:
        r = refs[0]
        print(f"Sample: {r.get("title", "(no title)")} ({r.get("year", "?")})")

Loaded 442 canonical bundles
Sample: Contrastive Personalized Exercise Recommendation With Reinforcement Learning (2024)


In [6]:
# --- Cell 3: Build summary tables (task labels discovered from data) ---
def my_label_fn(task_name):
    mapping = {"prediction": "PRED", "description": "DESC",
               "knowledge tracing": "KT", "recommendation": "REC"}
    return mapping.get(task_name.lower(), task_name)

dfs = build_summary_dataframes(bundles, task_label_fn=my_label_fn)
for name, df in dfs.items():
    print(f"  {name}: {len(df)} rows x {len(df.columns)} cols")
dfs.get("Reading List")

  Reading List: 442 rows x 15 cols


,id,title,authors,year,journal,doi,url,zotero_key,abstract,item_type,tags,provenance,validation_status,sync_metadata,page_id
0,003e69cb-f908-470d-bab2-c26eed3c63d3,Contrastive Personalized Exercise Recommendati...,[Wu et al.],2024,IEEE Transactions on Learning Technologies,,,,,,[],{'source_id': '003e69cb-f908-470d-bab2-c26eed3...,unknown,{},003e69cb-f908-470d-bab2-c26eed3c63d3
1,00f24000-f241-438b-a023-d872bc2b59e6,Using Clickstream Data Mining Techniques to Un...,[Rodriguez et al.],2021,LAK,,,,,,[],{'source_id': '00f24000-f241-438b-a023-d872bc2...,unknown,{},00f24000-f241-438b-a023-d872bc2b59e6
2,02eedcf7-bc5a-406e-ba94-01f7e39a25cd,Enhancing educational evaluation through predi...,[Xuan Lam et al],2024,Computers & Education: Artificial Intelligence,,,,,,[],{'source_id': '02eedcf7-bc5a-406e-ba94-01f7e39...,unknown,{},02eedcf7-bc5a-406e-ba94-01f7e39a25cd
3,02fef9f2-c5c2-47ee-a111-eb7788556c5e,Implementing AutoML in Educational Data Mining...,[Tsiakmaki et al.],2019,Applied Sciences,,,,,,[],{'source_id': '02fef9f2-c5c2-47ee-a111-eb77885...,unknown,{},02fef9f2-c5c2-47ee-a111-eb7788556c5e
4,033342f8-04d7-4a83-b56e-7d8b056b12e2,A Robust Machine Learning Technique to Predict...,[Liao et al.],2019,ACM Transactions on Computing Education,,,,,,[],{'source_id': '033342f8-04d7-4a83-b56e-7d8b056...,unknown,{},033342f8-04d7-4a83-b56e-7d8b056b12e2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
437,fbe0c99f-8e17-471c-a700-6586193253e2,Early-predicting dropout of university student...,[Cannistrà et al.],2022,Studies in Higher Education,,,,,,[],{'source_id': 'fbe0c99f-8e17-471c-a700-6586193...,unknown,{},fbe0c99f-8e17-471c-a700-6586193253e2
438,fce6f166-abcf-4029-b178-ef1e6fafd0db,Semi-Supervised Detection of Student Engagement,[Nezami et al.],2017,Twenty First Pacific Asia Conference on Inform...,,,,,,[],{'source_id': 'fce6f166-abcf-4029-b178-ef1e6fa...,unknown,{},fce6f166-abcf-4029-b178-ef1e6fafd0db
439,fd6153cf-0a1c-4664-9a99-74f4264d7d8e,A data-driven approach to predict first-year s...,[Gil et al.],2021,Education and Information Technologies,,,,,,[],{'source_id': 'fd6153cf-0a1c-4664-9a99-74f4264...,unknown,{},fd6153cf-0a1c-4664-9a99-74f4264d7d8e
440,fe76c297-6135-4a41-b537-d3637a4ac389,What are the most important predictors of comp...,[Hao et al.],2016,Computers in Human Behavior,,,,,,[],{'source_id': 'fe76c297-6135-4a41-b537-d3637a4...,unknown,{},fe76c297-6135-4a41-b537-d3637a4ac389


In [7]:
# --- Cell 4: Clean tables (caller-provided fix dicts) ---
MY_TYPO_FIXES = {r"Fiiltering": "Filtering", r"Exerrcise": "Exercise"}
MY_VALUE_MAP  = {"n/a": "Not Applicable", "none": "Not Applicable", "na": "Not Applicable"}
SEARCH_COLS   = ["Search Strategy"]

cleaned_dfs = {}
for name, df in dfs.items():
    cleaned, log = clean_table(df, typo_fixes=MY_TYPO_FIXES, value_map=MY_VALUE_MAP, search_strategy_columns=SEARCH_COLS)
    cleaned_dfs[name] = cleaned
    print(f"  {name}: {log["text_updates"]} cell(s) normalised")

  Reading List: 0 cell(s) normalised


In [8]:
# --- Cell 5: Shortcut — full pipeline ---
raw_dfs, clean_dfs, norm_log = run_analysis(
    pulled_dir,
    task_label_fn=my_label_fn,
    typo_fixes=MY_TYPO_FIXES,
    value_map=MY_VALUE_MAP,
    search_strategy_columns=SEARCH_COLS,
)
pd.DataFrame(norm_log).T

,rows_before,rows_after,text_updates,search_strategy_updates
Reading List,442,442,0,0


In [10]:
# --- Cell 6: Explore ---
rl = clean_dfs.get("Reading List", pd.DataFrame())
if not rl.empty and "year" in rl.columns:
    display(rl["year"].value_counts().sort_index())
if not rl.empty and "journal" in rl.columns:
    display(rl["journal"].dropna().value_counts().head(10))
for name, df in clean_dfs.items():
    if name != "Reading List" and not df.empty:
        print(f"{name} ({len(df)} rows):")
        display(df.head(5))

year
1995     1
2004     1
2008     1
2009     2
2010     3
2012     5
2013     5
2014     9
2015    15
2016    12
2017    29
2018    29
2019    25
2020    49
2021    57
2022    49
2023    75
2024    59
2025    14
2026     2
Name: count, dtype: int64

journal
LAK                                           54
IEEE Transactions on Learning Technologies    27
Computers & Education                         24
IEEE Access                                   15
Education and Information Technologies        14
Computers in Human Behavior                   14
Knowledge-Based Systems                       13
Applied Sciences                              11
ArXiv                                         11
Internet and Higher Education                 10
Name: count, dtype: int64